# Experimental Framework: Per-Dataset Model Benchmarking — Dataset 3
**Triage IQ — Predicting Hypertension from Emergency Department Triage Data**

This notebook implements the Phase 1 benchmarking pipeline for **Dataset 3**, which contains a clinically recorded hypertension label (`hypertension`). Because the dataset is class-imbalanced, a balanced sample is drawn by taking all positive cases and an equal number of negative cases before modelling.

Seven classifiers are evaluated using `RandomizedSearchCV` with 5-fold stratified cross-validation. SMOTE is applied within each fold to handle residual imbalance without leaking into the test set.

**Dataset:** `kaggle_data/dataset3.csv`  
**Target variable:** `hypertension` (clinically recorded, binary)


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import os

from scipy.stats import randint, uniform, mannwhitneyu, chi2_contingency

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (accuracy_score, roc_auc_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


## 2. Load Dataset and Define Column Groups

The `Unnamed: 0` index column is dropped on load. Column groups are defined upfront for consistent use throughout the notebook.


In [ ]:
df3 = pd.read_csv("kaggle_data/dataset3.csv")
df3 = df3.drop(columns=['Unnamed: 0'])

print("Dataset shape:", df3.shape)
print("\nColumn list:")
print(df3.columns.tolist())


In [ ]:
target = 'hypertension'

# Numeric features
numerical_cols = [
    'age', 'blood pressure', 'cholesterol', 'max heart rate',
    'plasma glucose', 'skin_thickness', 'insulin', 'bmi', 'diabetes_pedigree'
]

# Categorical features
categorical_cols = [
    'gender', 'chest pain type', 'exercise angina',
    'Residence_type', 'smoking_status', 'triage'
]

# Binary indicator features
binary_indicator_cols = ['heart_disease']

X = df3.drop(columns=[target])
y = df3[target]

print(f"Numeric features:          {len(numerical_cols)}")
print(f"Categorical features:      {len(categorical_cols)}")
print(f"Binary indicator features: {len(binary_indicator_cols)}")
print(f"\nClass distribution (full dataset):")
print(y.value_counts())


## 3. Balanced Sampling

The dataset is class-imbalanced. A balanced sample is created by taking all minority-class (hypertensive) records and an equal number of randomly drawn majority-class records, then shuffling.


In [ ]:
df_pos = df3[df3[target] == 1]
df_neg = df3[df3[target] == 0]

min_class_size = min(len(df_pos), len(df_neg))

df_sampled = pd.concat([
    df_pos.sample(n=min_class_size, random_state=42),
    df_neg.sample(n=min_class_size, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

print("Sampled dataset shape:", df_sampled.shape)
print("\nClass distribution:")
print(df_sampled[target].value_counts())
print("\nClass proportions:")
print(df_sampled[target].value_counts(normalize=True).round(3))


## 4. Statistical Feature Analysis

Three tests assess the association between each feature and the hypertension label:
- **Spearman correlation** — monotonic association for numeric features
- **Mann–Whitney U test** — non-parametric group comparison
- **Chi-square test** — association for categorical features


In [ ]:
# Spearman correlation (numeric features)
corr_df  = df_sampled[numerical_cols + [target]].corr(method='spearman')
htn_corr = corr_df[target].drop(target).sort_values(key=abs, ascending=False)

spearman_df = htn_corr.reset_index()
spearman_df.columns = ['Feature', 'Spearman Correlation']

print("Spearman Correlation with Hypertension:")
display(spearman_df)

plt.figure(figsize=(6, 8))
plt.barh(spearman_df['Feature'], spearman_df['Spearman Correlation'])
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel("Spearman Correlation")
plt.title("Spearman Correlation with Hypertension — Dataset 3")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Mann–Whitney U test (numeric features)
num_test_results = []
for col in numerical_cols:
    group0 = df_sampled[df_sampled[target] == 0][col].dropna()
    group1 = df_sampled[df_sampled[target] == 1][col].dropna()
    if len(group0) > 20 and len(group1) > 20:
        stat, p = mannwhitneyu(group0, group1, alternative='two-sided')
        num_test_results.append({"Feature": col, "p_value": p})

num_test_df = pd.DataFrame(num_test_results).sort_values("p_value")

print("Mann–Whitney U Test Results (Numeric Features):")
display(num_test_df)

plt.figure(figsize=(6, 8))
plt.barh(num_test_df['Feature'], -np.log10(num_test_df['p_value']))
plt.xlabel("-log10(p-value)")
plt.title("Mann–Whitney U Test — Dataset 3")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Chi-square test (categorical features)
cat_test_results = []
for col in categorical_cols:
    contingency = pd.crosstab(df_sampled[col], df_sampled[target])
    if contingency.shape[0] > 1:
        chi2, p, _, _ = chi2_contingency(contingency)
        cat_test_results.append({"Feature": col, "Chi2": chi2, "p_value": p})

cat_test_df = pd.DataFrame(cat_test_results).sort_values("p_value")

print("Chi-Square Test Results (Categorical Features):")
display(cat_test_df)

plt.figure(figsize=(6, 8))
plt.barh(cat_test_df['Feature'], -np.log10(cat_test_df['p_value']))
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel("-log10(p-value)")
plt.title("Chi-Square Test — Dataset 3")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 5. Pre-Model Feature Importance (XGBoost)

A preliminary XGBoost model is fitted on numeric features only to rank their importance before hyperparameter tuning begins. No pipeline or scaling is used here — this is a quick exploratory step only.


In [ ]:
X_pm = df_sampled[numerical_cols]
y_pm = df_sampled[target]

xgb_pipe_pm = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("xgb", XGBClassifier(
        n_estimators=100, max_depth=4, learning_rate=0.1,
        eval_metric="logloss", random_state=42
    ))
])
xgb_pipe_pm.fit(X_pm, y_pm)

importance_df = pd.DataFrame({
    "Feature":    numerical_cols,
    "Importance": xgb_pipe_pm.named_steps["xgb"].feature_importances_
}).sort_values("Importance", ascending=False)

print("Pre-Model Feature Importance (XGBoost — numeric features only):")
display(importance_df)

plt.figure(figsize=(7, 5))
plt.barh(importance_df["Feature"][::-1], importance_df["Importance"][::-1])
plt.xlabel("Feature Importance Score")
plt.title("Pre-Model Feature Importance — XGBoost (Dataset 3)")
plt.tight_layout()
plt.show()


## 6. Train/Test Split

A 70/30 stratified split is applied on the balanced sample. SMOTE is applied **within** each pipeline fold only — the test set remains unmodified throughout.


In [ ]:
X = df_sampled.drop(columns=[target])
y = df_sampled[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"\nTraining class distribution:")
print(y_train.value_counts())


## 7. Preprocessing Pipeline

Three column groups are handled separately:
- **Numeric** — median imputation + standard scaling
- **Categorical** — mode imputation + one-hot encoding
- **Binary indicators** — mode imputation only (already 0/1)


In [ ]:
preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler())
    ]), numerical_cols),

    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot',  OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_cols),

    ('bin', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent'))
    ]), binary_indicator_cols)
])


## 8. Model and Hyperparameter Grid Definitions

In [ ]:
models = {
    "LogisticRegression": (
        LogisticRegression(max_iter=2000),
        {"clf__C": uniform(0.01, 10), "clf__solver": ["liblinear", "saga"]}
    ),

    "DecisionTree": (
        DecisionTreeClassifier(),
        {
            "clf__max_depth":         randint(3, 6),
            "clf__min_samples_split": randint(5, 10),
            "clf__min_samples_leaf":  randint(2, 5)
        }
    ),

    "RandomForest": (
        RandomForestClassifier(),
        {
            "clf__n_estimators":      randint(100, 500),
            "clf__max_depth":         [None, 5, 10, 20],
            "clf__min_samples_split": randint(2, 20),
            "clf__min_samples_leaf":  randint(1, 10)
        }
    ),

    "SVM": (
        SVC(probability=True),
        {
            "clf__C":      uniform(0.1, 20),
            "clf__kernel": ["rbf"],
            "clf__gamma":  ["scale", "auto"]
        }
    ),

    "GradientBoosting": (
        GradientBoostingClassifier(),
        {
            "clf__n_estimators":  randint(50, 200),
            "clf__learning_rate": uniform(0.01, 0.2)
        }
    ),

    "MLP": (
        MLPClassifier(max_iter=2000),
        {
            "clf__hidden_layer_sizes": [(50,), (100,), (100, 50)],
            "clf__alpha":              uniform(0.0001, 0.01)
        }
    ),

    "XGBoost": (
        XGBClassifier(eval_metric='logloss', use_label_encoder=False),
        {
            "clf__n_estimators":     randint(100, 400),
            "clf__max_depth":        randint(3, 10),
            "clf__learning_rate":    uniform(0.01, 0.2),
            "clf__subsample":        uniform(0.6, 0.4),
            "clf__colsample_bytree": uniform(0.6, 0.4),
            "clf__gamma":            uniform(0, 5)
        }
    ),
}


## 9. Baseline Model Evaluation (No Hyperparameter Tuning)

Each model is trained once with default parameters to establish a performance floor before tuning.


In [ ]:
baseline_results = []

for name, (model, _) in models.items():
    print(f"Training baseline {name}...")

    pipe = ImbPipeline([
        ("preprocessor", preprocessor),
        ("smote",        SMOTE(random_state=42)),
        ("clf",          model)
    ])
    pipe.fit(X_train, y_train)

    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe["clf"], "predict_proba") else None

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba) if y_proba is not None else None

    baseline_results.append({
        "Model":            name,
        "Baseline Accuracy": round(acc, 3),
        "Baseline ROC-AUC":  round(auc, 3) if auc else None
    })
    print(f"  Accuracy: {acc:.3f}  |  ROC-AUC: {auc:.3f}\n")

baseline_df = pd.DataFrame(baseline_results).sort_values("Baseline ROC-AUC", ascending=False)
print("Baseline Results Summary:")
display(baseline_df)


## 10. Hyperparameter Tuning — RandomizedSearchCV

`RandomizedSearchCV` with 5-fold stratified cross-validation is run for each model.  
SMOTE is applied inside each fold. Optimisation metric: **ROC-AUC** (30 iterations per model).


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results     = []
best_models = {}

for name, (model, param_grid) in models.items():
    print(f"\nRunning RandomizedSearchCV for {name}...")

    pipe = ImbPipeline([
        ("preprocessor", preprocessor),
        ("smote",        SMOTE(random_state=42)),
        ("clf",          model)
    ])

    search = RandomizedSearchCV(
        pipe,
        param_distributions=param_grid,
        n_iter=30,
        scoring="roc_auc",
        cv=cv,
        random_state=42,
        verbose=0,
        n_jobs=-1
    )
    search.fit(X_train, y_train)
    best_models[name] = search

    best_pipe = search.best_estimator_
    y_pred    = best_pipe.predict(X_test)
    y_proba   = (
        best_pipe.predict_proba(X_test)[:, 1]
        if hasattr(best_pipe["clf"], "predict_proba") else None
    )

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba) if y_proba is not None else None

    results.append({
        "Model":                              name,
        "Best CV Score (RandomizedSearchCV)": round(search.best_score_, 3),
        "CV Accuracy":                        round(acc, 3),
        "CV ROC-AUC":                         round(auc, 3) if auc else None
    })

    print(f"  Best CV ROC-AUC: {search.best_score_:.3f}  |  "
          f"Accuracy: {acc:.3f}  |  ROC-AUC: {auc:.3f}")
    print(classification_report(y_test, y_pred))
    print("-" * 70)

results_df = pd.DataFrame(results).sort_values("CV ROC-AUC", ascending=False)
print("\nModel Performance Summary — Dataset 3:")
display(results_df)


## 11. Before vs After Tuning Comparison

In [ ]:
comparison_df = pd.merge(
    baseline_df,
    results_df[['Model', 'CV Accuracy', 'CV ROC-AUC']],
    on="Model"
).rename(columns={
    "CV Accuracy": "Tuned Accuracy",
    "CV ROC-AUC":  "Tuned ROC-AUC"
})

print("Before vs After Tuning Comparison:")
display(comparison_df)


## 12. Model Performance Visualisation

In [ ]:
# Summary table plot
display_df = results_df[['Model', 'CV Accuracy', 'Best CV Score (RandomizedSearchCV)', 'CV ROC-AUC']].copy()
display_df = display_df.rename(columns={'Best CV Score (RandomizedSearchCV)': 'Best CV Score'})

fig, ax = plt.subplots(figsize=(10, len(display_df) * 0.5 + 1))
ax.axis('off')
tbl = ax.table(
    cellText=display_df.values,
    colLabels=display_df.columns,
    cellLoc='center', loc='center'
)
for j in range(len(display_df.columns)):
    tbl[0, j].set_facecolor('#40466e')
    tbl[0, j].set_text_props(color='w', weight='bold')
for i in range(1, len(display_df) + 1):
    color = '#f9f9f9' if i % 2 == 1 else '#ffffff'
    for j in range(len(display_df.columns)):
        tbl[i, j].set_facecolor(color)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.5)
plt.title("Model Performance — Dataset 3 (RandomizedSearchCV)", fontsize=12, weight='bold')
plt.tight_layout()
plt.show()

# Bar chart
plot_df = results_df.melt(
    id_vars='Model',
    value_vars=['CV Accuracy', 'CV ROC-AUC'],
    var_name='Metric', value_name='Score'
)
plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x='Model', y='Score', hue='Metric', palette='Blues_d')
plt.title("Model Performance Comparison — Dataset 3")
plt.ylim(0.6, 1.05)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()


## 13. Best Hyperparameters (RandomizedSearchCV)

In [ ]:
best_params_list = []
for model_name, search in best_models.items():
    for param, value in search.best_params_.items():
        best_params_list.append({
            "Model":      model_name,
            "Parameter":  param.replace("clf__", ""),
            "Best Value": value
        })

best_params_df = pd.DataFrame(best_params_list)

bp_display = best_params_df.copy()
bp_display['Model_display'] = bp_display['Model']
bp_display.loc[
    bp_display['Model_display'] == bp_display['Model_display'].shift(1),
    'Model_display'
] = ''

fig, ax = plt.subplots(figsize=(12, len(bp_display) * 0.35 + 2))
ax.axis('off')
tbl = ax.table(
    cellText=bp_display[['Model_display', 'Parameter', 'Best Value']].values,
    colLabels=['Model', 'Parameter', 'Best Value'],
    cellLoc='center', loc='center'
)
for j in range(3):
    tbl[0, j].set_facecolor('#40466e')
    tbl[0, j].set_text_props(color='w', weight='bold')
for i in range(1, len(bp_display) + 1):
    color = '#f9f9f9' if i % 2 == 1 else '#ffffff'
    for j in range(3):
        tbl[i, j].set_facecolor(color)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.1, 1.2)
plt.title("RandomizedSearchCV Best Parameters — Dataset 3", fontsize=12, weight='bold')
plt.tight_layout()
plt.show()

display(best_params_df)


## 14. Confusion Matrix — XGBoost (Best Model)

The confusion matrix for the best-performing XGBoost model on the held-out test set.


In [ ]:
xgb_best_pipe  = best_models["XGBoost"].best_estimator_
y_pred_xgb     = xgb_best_pipe.predict(X_test)

cm = confusion_matrix(y_test, y_pred_xgb)
print("Confusion Matrix (XGBoost):")
print(cm)

fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Hypertension", "Hypertension"]
).plot(cmap="Blues", ax=ax, colorbar=False)
plt.title("Confusion Matrix — XGBoost (Test Set, Dataset 3)")
plt.tight_layout()
plt.show()


## 15. SHAP Explainability — XGBoost (Dataset 3)

SHAP (SHapley Additive Explanations) identifies which features drive XGBoost predictions globally (summary + beeswarm) and locally (waterfall for a single sample).


In [ ]:
pipeline_df3 = best_models["XGBoost"].best_estimator_

# Transform training features through the preprocessing step only
X_transformed = pipeline_df3.named_steps['preprocessor'].transform(X_train)
if hasattr(X_transformed, "toarray"):
    X_transformed = X_transformed.toarray()

# Reconstruct feature names from all three column groups
cat_features_out = (
    pipeline_df3.named_steps['preprocessor']
    .named_transformers_['cat']
    .named_steps['onehot']
    .get_feature_names_out(categorical_cols)
)
feature_names = list(numerical_cols) + list(cat_features_out) + list(binary_indicator_cols)
print(f"Total features after encoding: {len(feature_names)}")


In [ ]:
explainer       = shap.TreeExplainer(pipeline_df3.named_steps['clf'])
raw_shap_values = explainer.shap_values(X_transformed)

shap_explanation = shap.Explanation(
    values=raw_shap_values,
    base_values=explainer.expected_value,
    data=X_transformed,
    feature_names=feature_names
)

# Global bar summary
print("Global SHAP Summary Plot (Dataset 3):")
shap.summary_plot(shap_explanation, plot_type='bar')


In [ ]:
# Beeswarm — direction of feature effects
print("SHAP Beeswarm Plot:")
shap.summary_plot(shap_explanation, X_transformed, feature_names=feature_names)


In [ ]:
# Local explanation — first training sample
print("Local SHAP Explanation (first training sample):")
shap.plots.waterfall(shap_explanation[0], max_display=10)


## 16. Save Results to Excel

In [ ]:
os.makedirs("results", exist_ok=True)

# Performance table
performance_file = "results/model_performance_dataset3.xlsx"
results_df.to_excel(performance_file, index=False)
print(f"Model performance saved: {performance_file}")

# Best parameters table
params_file = "results/best_params_dataset3.xlsx"
best_params_df.to_excel(params_file, index=False)
print(f"Best parameters saved:   {params_file}")
